# PropagandaTrace — Full Pipeline
**Phases 1 → 2 → 3**: Data Collection → Watermarked Corpus Generation → Evasion Attacks

**Before running:** Go to `Runtime → Change runtime type → GPU (T4)`

---
## Step 1 — Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → GPU (T4) and re-run.")

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1e9
print(f"GPU : {gpu.name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 14:
    print("WARNING: < 14 GB VRAM. 4-bit quantization will be used; generation may be slow.")
else:
    print("VRAM OK — 4-bit quantization enabled.")

---
## Step 2 — Mount Google Drive and upload project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# Upload your project zip from local machine.
# Run this cell, click 'Choose Files', and select PropagandaTrace.zip
from google.colab import files
uploaded = files.upload()   # upload PropagandaTrace.zip

In [ ]:
import os, zipfile

zip_name = [k for k in uploaded.keys() if k.endswith('.zip')][0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/propagandatrace')

# Find the project root (handles zip with or without top-level folder)
candidates = ['/content/propagandatrace']
for entry in os.listdir('/content/propagandatrace'):
    full = f'/content/propagandatrace/{entry}'
    if os.path.isdir(full) and os.path.exists(f'{full}/config.yaml'):
        candidates = [full]
        break

PROJECT = candidates[0]
os.chdir(PROJECT)
print(f"Working directory: {os.getcwd()}")
print(os.listdir('.'))

---
## Step 3 — Install dependencies

In [ ]:
%%bash
pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.27.0 \
    bitsandbytes>=0.43.0 \
    sentencepiece>=0.2.0 \
    protobuf \
    datasets>=2.18.0 \
    huggingface-hub \
    requests pandas numpy tqdm \
    pyyaml jsonlines
echo "All packages installed."

In [ ]:
# Verify core imports
import torch, transformers, datasets, bitsandbytes
print(f"torch        {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
print(f"transformers {transformers.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print(f"datasets     {datasets.__version__}")

---
## Step 4 — HuggingFace Login
Required for gated models: **Mistral-7B** and **LLaMA-3-8B**.

Get your token at https://huggingface.co/settings/tokens  
Make sure you have accepted the model licenses at:
- https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
- https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option A — store token in Colab Secrets (recommended, key name: HF_TOKEN)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in via Colab Secret.")
except Exception:
    # Option B — paste token directly (less secure)
    import getpass
    hf_token = getpass.getpass("Paste your HuggingFace token: ")
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in via manual input.")

---
## Step 5 — Create required directories

In [ ]:
import os
for d in ['data/raw', 'data/corpus', 'data/evasion', 'data/results']:
    os.makedirs(d, exist_ok=True)
print("Directories ready:", os.listdir('data'))

---
## Phase 1 — Data Collection
Downloads seed prompts from WTWT, GDELT, SemEval-2020, and templates.  
Output: `data/raw/seed_prompts.jsonl`

In [ ]:
!python phase1_collect_data.py --config config.yaml

In [ ]:
import json
from collections import Counter

with open('data/raw/collection_stats.json') as f:
    stats = json.load(f)

print("=== Phase 1 Results ===")
for k, v in stats.items():
    print(f"  {k:15s}: {v}")

# Preview first 3 prompts
print("\n--- Sample prompts ---")
with open('data/raw/seed_prompts.jsonl') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        r = json.loads(line)
        print(f"[{r['source']}] {r['text'][:120]}")

---
## Phase 2 — Watermarked Corpus Generation
Generates propaganda texts using 3 LLMs × 2 watermark schemes.  
Output: `data/corpus/<model>_<scheme>.jsonl`

**Full run** (~3 hrs on T4): 3 models × 2 schemes × 3,000 texts = 18,000 records  
**Quick test** (5–10 min): set `MAX_TEXTS = 50` below

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# Set to None for full run (3000 per model), or a small int to test
MAX_TEXTS = None   # e.g. 50 for a quick test
# ────────────────────────────────────────────────────────────────────

In [ ]:
# Run all 3 models × both schemes sequentially
# Each model is loaded, generates, then unloaded to free VRAM.

max_arg = f"--max_texts {MAX_TEXTS}" if MAX_TEXTS else ""

for model in ['mistral', 'llama', 'falcon']:
    for scheme in ['kgw', 'aaronson']:
        print(f"\n{'='*60}")
        print(f"  Model: {model}   Scheme: {scheme}")
        print(f"{'='*60}")
        !python phase2_generate_corpus.py \
            --config config.yaml \
            --model {model} \
            --scheme {scheme} \
            {max_arg}

In [ ]:
# Inspect Phase 2 output
import os, json
from pathlib import Path

corpus_files = sorted(Path('data/corpus').glob('*.jsonl'))
print(f"Corpus files: {len(corpus_files)}")
print()
for f in corpus_files:
    count = sum(1 for _ in open(f, encoding='utf-8'))
    print(f"  {f.name:35s} {count:5d} records")

# Preview one record
if corpus_files:
    print("\n--- Sample record ---")
    with open(corpus_files[0], encoding='utf-8') as fh:
        sample = json.loads(fh.readline())
    for k, v in sample.items():
        if k != 'token_ids':
            print(f"  {k:15s}: {str(v)[:100]}")

---
## Phase 3 — Evasion Attacks
Applies 4 evasion strategies to every corpus file:  
`t5`, `pegasus`, `rtt_arabic`, `rtt_russian`

Output: `data/evasion/<model>_<scheme>_evade_<strategy>.jsonl`

**Full run** (~2 hrs): all 6 corpus files × 4 strategies = 24 output files  
**Quick test**: set `MAX_RECORDS = 50` below

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
MAX_RECORDS = None   # e.g. 50 for a quick test
# ────────────────────────────────────────────────────────────────────

In [ ]:
max_arg = f"--max_records {MAX_RECORDS}" if MAX_RECORDS else ""

!python phase3_evasion.py \
    --config config.yaml \
    --strategy all \
    {max_arg}

In [ ]:
# Inspect Phase 3 output
from pathlib import Path
import json

evasion_files = sorted(Path('data/evasion').glob('*.jsonl'))
print(f"Evasion files: {len(evasion_files)}")
print()
for f in evasion_files:
    count = sum(1 for _ in open(f, encoding='utf-8'))
    print(f"  {f.name:55s} {count:5d} records")

# Preview one evaded record
if evasion_files:
    print("\n--- Sample evaded record ---")
    with open(evasion_files[0], encoding='utf-8') as fh:
        sample = json.loads(fh.readline())
    print(f"  strategy : {sample.get('evasion_strategy')}")
    print(f"  original : {sample.get('generated', '')[:120]}")
    print(f"  evaded   : {sample.get('evaded_text', '')[:120]}")

---
## Save results to Google Drive

In [ ]:
import shutil, os

dest = '/content/drive/MyDrive/PropagandaTrace_data'
os.makedirs(dest, exist_ok=True)

for folder in ['data/raw', 'data/corpus', 'data/evasion']:
    shutil.copytree(folder, f"{dest}/{folder.split('/')[-1]}", dirs_exist_ok=True)

print(f"All data saved to Google Drive: {dest}")
for sub in os.listdir(dest):
    files = os.listdir(f"{dest}/{sub}")
    print(f"  {sub}/  ({len(files)} files)")

---
## Dataset Summary

In [ ]:
from pathlib import Path
import json

def count_jsonl(path):
    return sum(1 for _ in open(path, encoding='utf-8'))

print("=" * 60)
print("PROPAGANDATRACE — DATASET SUMMARY")
print("=" * 60)

# Phase 1
stats = json.load(open('data/raw/collection_stats.json'))
print(f"\nPhase 1 — Seed prompts")
print(f"  Total : {stats['total']}")
print(f"  WTWT  : {stats['wtwt']}")
print(f"  GDELT : {stats['gdelt']}")
print(f"  SemEval: {stats['semeval2020']}")
print(f"  Templates: {stats['template']}")

# Phase 2
corpus_files = list(Path('data/corpus').glob('*.jsonl'))
total_corpus = sum(count_jsonl(f) for f in corpus_files)
print(f"\nPhase 2 — Watermarked corpus")
print(f"  Files : {len(corpus_files)}")
print(f"  Total : {total_corpus} records")
for f in sorted(corpus_files):
    print(f"  {f.name}: {count_jsonl(f)}")

# Phase 3
evasion_files = list(Path('data/evasion').glob('*.jsonl'))
total_evasion = sum(count_jsonl(f) for f in evasion_files)
print(f"\nPhase 3 — Evasion attacks")
print(f"  Files : {len(evasion_files)}")
print(f"  Total : {total_evasion} records")

print("\n" + "=" * 60)

---
## Phase 4 — Detection & Attribution Scoring
Runs `KGWDetector` and `AaronsonDetector` on every corpus and evasion file.
No GPU needed — pure scoring math on already-generated text.

Output: `data/results/`

In [ ]:
# Cell 4.1 — Install plotting deps
!pip install -q matplotlib seaborn

In [ ]:
# Cell 4.2 — Run Phase 4 (detection & attribution scoring)
# Set to None to score everything, or a small int for a quick test
MAX_RECORDS_P4 = None   # e.g. 100

import os
os.makedirs('data/results', exist_ok=True)

max_arg = f"--max_records {MAX_RECORDS_P4}" if MAX_RECORDS_P4 else ""
!python phase4_evaluate.py --config config.yaml {max_arg}

In [ ]:
# Cell 4.3 — Print paper-style detection rate tables
import pandas as pd
from pathlib import Path

results_dir = Path('data/results')

for scheme in ['kgw', 'aaronson']:
    for detector in ['kgw', 'aar']:
        fname = results_dir / f"table_{detector}_detection_{scheme}.csv"
        if fname.exists():
            df = pd.read_csv(fname, index_col=0)
            print(f"\n{'='*60}")
            print(f"  {detector.upper()} Detection Rate (%)  —  scheme: {scheme.upper()}")
            print(f"{'='*60}")
            print(df.to_string())

print(f"\n{'='*60}")
print("Full Metrics Summary")
print(f"{'='*60}")
summary = pd.read_csv(results_dir / 'metrics_summary.csv')
print(summary.to_string(index=False))

In [ ]:
# Cell 4.5 — Save Phase 4 results to Google Drive
import shutil
from pathlib import Path

dest = '/content/drive/MyDrive/PropagandaTrace_data'
shutil.copytree('data/results', f"{dest}/results", dirs_exist_ok=True)
print(f"Phase 4 results saved to: {dest}/results/")
for f in sorted(Path(f"{dest}/results").rglob('*')):
    if f.is_file():
        print(f"  {f.relative_to(dest)}")

In [ ]:
# Cell 4.4 — Display all plots inline
from pathlib import Path
from IPython.display import display, Image

plot_dir = Path('data/results/plots')
plot_files = sorted(plot_dir.glob('*.png'))
print(f"Plots generated: {len(plot_files)}")
for p in plot_files:
    print(f"\n── {p.name} ──")
    display(Image(str(p)))